# **Exploración de los datos**
---

## **1. Librerías**
---

In [2]:
# Importar librerías
import sys
from pathlib import Path

# Añadir el directorio raíz del proyecto al path de Python PRIMERO
proyecto_root = str(Path('.').resolve().parent)
if proyecto_root not in sys.path:
    sys.path.insert(0, proyecto_root)

import pandas as pd
import os
import argparse 
from datetime import datetime
from dateutil import parser
import matplotlib.pyplot as plt
import numpy as np

# Recargar módulo para obtener las funciones más recientes
import importlib
if 'src.data.funciones_parte_1' in sys.modules:
    importlib.reload(sys.modules['src.data.funciones_parte_1'])

from src.data.funciones_parte_1 import convertir_parquet, leer_parquet_si_existe

## **2. Análisis de la base de datos**
---

### **2.1 Cargue de la base de datos**
---

In [4]:
# Definir rutas relativas
ruta_csv = os.path.join(proyecto_root, 'data', 'original', 'card_transaction.csv')
ruta_parquet = os.path.join(proyecto_root, 'data', 'processed', 'card_transaction.parquet')

# Verificar si el archivo Parquet existe, si no, convertir
if not os.path.exists(ruta_parquet):
    print(f"Convirtiendo CSV a Parquet")
    convertir_parquet(ruta_csv, ruta_parquet, chunksize=2_000_000)
    print(f"Conversión completada\n")
else:
    print(f"Archivo ya existe, no necesito crearlo\n")

# Cargar el archivo Parquet
df_cards = pd.read_parquet(ruta_parquet)
print(f"Dimensiones del dataset: {df_cards.shape}")

Convirtiendo CSV a Parquet
Iniciando conversión de C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\original\card_transaction.csv a C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\processed\card_transaction.parquet
Procesando en chunks de 2000000 filas
Conversión exitosa: 24,386,900 filas guardadas en C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\processed\card_transaction.parquet
Tamaño original: 2.19 GB
Tamaño Parquet: 0.19 GB
Conversión completada

Dimensiones del dataset: (24386900, 15)


In [13]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
print(f"Ver DataFrame:")
pd.concat([df_cards.head(2), df_cards.tail(2)])

Ver DataFrame:


,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors?,is_fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
24386898,1999,1,2020,2,28,20:10,$43.12,Chip Transaction,2500998799892805156,Merrimack,NH,3054.0,4121,NaN,No
24386899,1999,1,2020,2,28,23:10,$45.13,Chip Transaction,4751695835751691036,Merrimack,NH,3054.0,5814,NaN,No


In [6]:
# Leer si existe
df_cards = leer_parquet_si_existe(ruta_parquet)

Archivo cargado: C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\processed\card_transaction.parquet
 Dimensiones: 24,386,900 filas × 15 columnas


### **2.2 Análisis de la base de datos**
---

In [7]:
# Cambio el nombre de las columnas
df_cards.columns=df_cards.columns.str.strip().str.lower().str.replace(' ', '_')
df_cards.columns

Index(['user', 'card', 'year', 'month', 'day', 'time', 'amount', 'use_chip',
       'merchant_name', 'merchant_city', 'merchant_state', 'zip', 'mcc',
       'errors?', 'is_fraud?'],
      dtype='str')

In [15]:
# Verificar valores nulos
for i in df_cards.columns:
    print(f"El número de valores nulos para la columna: {i}")
    print(f" es de: {df_cards[i].isnull().sum()}")

El número de valores nulos para la columna: user
 es de: 0
El número de valores nulos para la columna: card
 es de: 0
El número de valores nulos para la columna: year
 es de: 0
El número de valores nulos para la columna: month
 es de: 0
El número de valores nulos para la columna: day
 es de: 0
El número de valores nulos para la columna: time
 es de: 0
El número de valores nulos para la columna: amount
 es de: 0
El número de valores nulos para la columna: use_chip
 es de: 0
El número de valores nulos para la columna: merchant_name
 es de: 0
El número de valores nulos para la columna: merchant_city
 es de: 0
El número de valores nulos para la columna: merchant_state
 es de: 2720821
El número de valores nulos para la columna: zip
 es de: 2878135
El número de valores nulos para la columna: mcc
 es de: 0
El número de valores nulos para la columna: errors?
 es de: 23998469
El número de valores nulos para la columna: is_fraud?
 es de: 0


In [12]:
# Mirar la distribución de las variables
for i in df_cards.columns:
    if df_cards[i].dtype == 'object':
        print(f"Columna: {i}")
        print(df_cards[i].nunique())

In [ ]:
# Validar duplicados en las columnas de cuenta de df_tranx
print("Columnas en df_tranx:", df_tranx.columns.tolist())
print("\nInterpretación probable:")
print("- 'account' es la cuenta de origen asociada a from_bank")
print("- 'account.1' es la cuenta de destino asociada a to_bank")
print("- Por eso hay 2 columnas: una cuenta de envío y otra cuenta de recepción\n")

for col, label in [('account', 'origen'), ('account.1', 'destino')]:
    if col in df_tranx.columns:
        total = len(df_tranx)
        uniques = df_tranx[col].nunique(dropna=False)
        duplicates = df_tranx[col].duplicated().sum()
        print(f"{col} ({label}) -> total: {total:,}, únicos: {uniques:,}, duplicados: {duplicates:,}")
        if duplicates > 0:
            print(f"Valores repetidos en {col} (top 10):")
            display(df_tranx[col].value_counts().loc[lambda x: x > 1].head(10))
        print("")
    else:
        print(f"No existe la columna {col} en df_tranx")

# Validar filas únicas por par de cuentas origen/destino
df_tranx['pair_account'] = df_tranx['from_bank'].astype(str) + '|' + df_tranx['account'].astype(str) + ' -> ' + df_tranx['to_bank'].astype(str) + '|' + df_tranx['account.1'].astype(str)
unique_pairs = df_tranx['pair_account'].nunique(dropna=False)
duplicate_pair_rows = df_tranx['pair_account'].duplicated().sum()
print(f"Par origen-destino -> únicos: {unique_pairs:,}, filas duplicadas: {duplicate_pair_rows:,}")
# Borrar columna temporal si no se necesita
df_tranx.drop(columns=['pair_account'], inplace=True)


Columnas en df_tranx: ['timestamp', 'from_bank', 'account', 'to_bank', 'account.1', 'amount_received', 'receiving_currency', 'amount_paid', 'payment_currency', 'payment_format', 'is_laundering']

Interpretación probable:
- 'account' es la cuenta de origen asociada a from_bank
- 'account.1' es la cuenta de destino asociada a to_bank
- Por eso hay 2 columnas: una cuenta de envío y otra cuenta de recepción

account (origen) -> total: 179,702,229, únicos: 2,067,987, duplicados: 177,634,242
Valores repetidos en account (top 10):


account
100428660    6233962
1004286A8    3931462
1004286F0    1207403
1004289C0     769178
100428858     592485
100428810     555522
1004287C8     526263
1004288A0     508666
100428738     466909
100428978     460175
Name: count, dtype: int64


account.1 (destino) -> total: 179,702,229, únicos: 1,717,142, duplicados: 177,985,087
Valores repetidos en account.1 (top 10):


account.1
100428660    31906
1004286A8    20073
1004286F0     7164
1004289C0     3981
100428858     2968
1004288A0     2727
100428810     2591
1004287C8     2554
100428978     2305
100428738     2282
Name: count, dtype: int64


Par origen-destino -> únicos: 8,466,789, filas duplicadas: 171,235,440


: 

In [26]:
print(f"Dado que el número de filas del conjunto de datos es de:{df_accounts.shape[0]:,}.") 
print(f"y el número de cuentas únicas es de: {df_accounts['account_number'].nunique():,}.  Se concluye que hay: {df_accounts.shape[0] - df_accounts['account_number'].nunique():,} cuentas repetidas.")

NameError: name 'df_accounts' is not defined

In [8]:
# Creo una llave unica para cada cuenta combinando bank_id y account_number
# No hay valores duplicados después de crear la llave única, lo que es una buena señal.
df_accounts["account_key"]=df_accounts["bank_id"].astype(str)+"-"+df_accounts["account_number"].astype(str)
duplicados_cuenta_key=df_accounts['account_key'].value_counts()
if duplicados_cuenta_key[duplicados_cuenta_key>1].empty:
    print("No hay cuentas duplicadas después de crear la llave única.")
else:
    print("Hay cuentas duplicadas después de crear la llave única:")
    print(duplicados_cuenta_key[duplicados_cuenta_key>1])

No hay cuentas duplicadas después de crear la llave única.


In [9]:
# tipo de corporacion
df_accounts['entity_type'] = df_accounts['entity_name'].str.split('#').str[0].str.strip()
df_accounts["entity_type"].value_counts()

entity_type
Partnership            745341
Corporation            692680
Sole Proprietorship    680013
Country                  5689
Individual               3104
Direct                     28
Name: count, dtype: int64

In [10]:
# Las entidades pueden tener más de 1 banco adscrito
cuentas_por_entidad = df_accounts.groupby('entity_id')['bank_id'].nunique().reset_index()
cuentas_por_entidad.columns = ['entity_id', 'num_bancos_distintos']
cuentas_por_entidad.head(3)

,entity_id,num_bancos_distintos
0,2AA02E7E570,2
1,2AA02E7E5F0,15
2,2AA02E7E650,2


In [11]:
# Unir esto al DataFrame original
df_accounts = df_accounts.merge(cuentas_por_entidad, on='entity_id', how='left')

In [12]:
pd.concat([df_accounts.head(2), df_accounts.tail(2)])
df_accounts.head()

,bank_name,bank_id,account_number,entity_id,entity_name,account_key,entity_type,num_bancos_distintos
0,Portugal Bank #500,240522,82655C500,2AA04EEC5D0,Corporation #82502,240522-82655C500,Corporation,10
1,First Bank of Laramie,339367,8505ED380,2AA06B8AC80,Partnership #15630,339367-8505ED380,Partnership,21
2,National Bank of Helena,368763,826283D80,2AA066499F0,Corporation #12918,368763-826283D80,Corporation,18
3,Switzerland Bank #2454,3174937,842090C80,2AA06712690,Corporation #135403,3174937-842090C80,Corporation,477
4,China Bank #561,53267,817D00980,2AA053417D0,Corporation #103595,53267-817D00980,Corporation,15


### **2.3 Exportar la base de datos**
---

In [13]:
ruta_data_limpio = 'C:\\Users\\cecor\\OneDrive - Universidad Nacional de Colombia\\GitHub\\MSc-Thesis-Financial-Fraud-Detection-Models\\data\\processed'
archivo_cuentas = 'HI-Large_accounts_limpio'
ruta_archivo = os.path.join(ruta_data_limpio, archivo_cuentas + '.parquet')
#
try:
    # index=False para no columna extra
    df_accounts.to_parquet(ruta_archivo, index=False, engine='pyarrow')
    print(f"¡Archivo exportado exitosamente en:\n{ruta_archivo}")    
except Exception as e:
    print(f"Ocurrió un error al exportar: {e}")

¡Archivo exportado exitosamente en:
C:\Users\cecor\OneDrive - Universidad Nacional de Colombia\GitHub\MSc-Thesis-Financial-Fraud-Detection-Models\data\processed\HI-Large_accounts_limpio.parquet
